In [1]:
from src.clp_zne.pauli_noise_sim import clp_zne_under_pauli_noise
from src.clp_zne.hamiltonians import sherrington_kirkpatrick_model
from src.clp_zne.utils import compute_evals_ideal
from qiskit.circuit.library import TwoLocal
from qiskit import transpile
import numpy as np
from tqdm import tqdm

In [2]:
OUTPUT_FOLDER = r"data\FakeTorino--Cyclic circuit--12 qubits--Pauli noise--CLP ZNE 3 params 4 cycles"
N_QUBITS = 12
N_CIRCUITS = 20
N_LAYERS = 3
N_OBSERVABLES = 100

In [3]:
observables = [sherrington_kirkpatrick_model(N_QUBITS, h=1, seed=i) for i in range(N_OBSERVABLES)]

circuits = []
for i in range(N_CIRCUITS):
    circ = TwoLocal(N_QUBITS, ['rx', 'rz'], 'cz', entanglement='circular', reps=N_LAYERS)
    rng = np.random.default_rng(i)
    parameters = rng.uniform(-np.pi, np.pi, circ.num_parameters)
    circ.assign_parameters(parameters, inplace=True)
    circuits.append(circ)

In [ ]:
all_evals_ideal = []
all_evals_mitigated = []
all_evals_noisy = []
all_error_sums = []

for circ in tqdm(circuits):
    # Transpile the circuit
    tcirc = transpile(circ, basis_gates=['u1', 'u2', 'u3', 'cz'], optimization_level=1)
    
    # Perform error mitigation
    evals_mitigated, evals_noisy, error_sums = clp_zne_under_pauli_noise(tcirc, observables)
    
    # Compute noiseless expectation value
    evals_ideal = compute_evals_ideal(circ, observables)

    
    all_evals_ideal.append(evals_ideal)
    all_evals_mitigated.append(evals_mitigated)
    all_evals_noisy.append(evals_noisy)
    all_error_sums.append(error_sums)

all_evals_ideal = np.array(all_evals_ideal)
all_evals_mitigated = np.array(all_evals_mitigated)
all_evals_noisy = np.array(all_evals_noisy)
all_error_sums = np.array(all_error_sums) 

In [26]:
# Save the results
data_to_save = {
    'evals_ideal.npy': all_evals_ideal,
    'evals_mitigated.npy': all_evals_mitigated_cf,
    'evals_noisy.npy': all_evals_noisy_cf,
    'error_sums.npy': all_error_sums
}

for filename, data in data_to_save.items():
    path = os.path.join(OUTPUT_FOLDER, filename)
    np.save(path, data)

print(f"Successfully saved {len(data_to_save)} files to: {OUTPUT_FOLDER}")

Successfully saved 3 files to: .\data\upd Pauli noise 12 qubits exp cf
